# Milestone 1 (GPU) — retrieval methods that actually generalise

The hand-written synonym table did **not** generalise: +71.9 points on the 32
records whose failures were audited, **+0.9** on the 118 never inspected. You
cannot enumerate how citizens phrase legal questions.

This notebook tests the three fixes that work *without* having seen the phrasing:

1. **Cross-encoder reranking** — reads question and statute together
2. **Stronger multilingual embeddings** — current e5-base adds only +2pts over BM25
3. **HyDE query expansion** — the model writes statutory phrasing itself

Every result is judged on `full_hit_never_audited`. The tuned-on column is
printed too, purely so the gap between them stays visible.

**Settings:** GPU **T4 x2**, Internet **On**. ~1–2h.

In [ ]:
# --- setup + preflight -------------------------------------------------
!pip -q install -U transformers accelerate sentence-transformers rank_bm25

import os, subprocess, sys, time
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch

def run(cmd):
    """Fail loudly. `!python` swallows non-zero exits and hides the real error."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    if subprocess.run([str(c) for c in cmd], text=True).returncode != 0:
        raise RuntimeError(f"step failed: {' '.join(str(c) for c in cmd)}")

REPO = "https://github.com/JitendraJha98/nyaya-model.git"
if not os.path.exists("/kaggle/working/nyaya-model"):
    subprocess.run(["git", "clone", "--depth", "1", REPO,
                    "/kaggle/working/nyaya-model"], check=True)
os.chdir("/kaggle/working/nyaya-model")
sys.path.insert(0, "src")

# Kaggle can allocate a P100 (sm_60) that the preinstalled torch cannot use.
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
print(f"GPU: {torch.cuda.get_device_name(0)} ({arch})")
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError(f"{arch} unsupported by this torch build "
                       f"({torch.cuda.get_arch_list()}). Switch to T4 x2.")
print("preflight OK")

In [ ]:
# --- smoke: prove the reranker runs here, and time it ------------------
# Runs 1-4 of the previous notebook died on environment assumptions. Five
# minutes here is cheaper than discovering it 90 minutes in.
sys.path.insert(0, "src")
from nyaya.rerank import CrossEncoderReranker

rr = CrossEncoderReranker(depth=50, batch_size=32)
passages = ["BSA Section 63: Admissibility of electronic records. " + "x " * 150] * 32
query = "Are call detail records admissible?"

# WARM UP FIRST. The first score() call lazily downloads ~2.3 GB and loads the
# model; timing it reported 1081 ms/pair on a T4 and tripped this very guard.
# Measure steady-state inference, not the download.
t0 = time.time()
rr.score(query, passages[:4])
print(f"model download + load: {time.time() - t0:.0f}s")

t0 = time.time()
rr.score(query, passages)
per_pair = (time.time() - t0) / len(passages)
print(f"device={rr.device}  {per_pair*1000:.0f} ms/pair (steady state)")

# Project the REAL workload. An earlier version counted only the 150
# gold-bearing records and so under-projected by 5.7x: retrieve() is called
# once per eval record in the recall loop (500) AND once per no-gold record in
# phrase_coverage (~350), and every call scores `depth` pairs.
DEPTH, RETRIEVE_CALLS, N_RERANK_CONFIGS = 50, 850, 2
pairs = RETRIEVE_CALLS * DEPTH
projected = per_pair * pairs * N_RERANK_CONFIGS / 60
print(f"{pairs:,} pairs/config x {N_RERANK_CONFIGS} reranked configs")
print(f"projected reranked sweeps: ~{projected:.0f} min")
if projected > 150:
    raise RuntimeError(f"{projected:.0f} min exceeds budget — reduce "
                       f"--rerank-depth or drop a config before burning quota.")
print("smoke OK")

## Baselines and the three candidate methods

Each writes its own report file, so no run can overwrite another's numbers.

In [ ]:
# Baseline: BM25 + synonyms (no GPU) — the number to beat
run([sys.executable, "scripts/15_retrieval_recall.py", "--k", 1, 3, 5, 8])

In [ ]:
# Method 1: cross-encoder reranking (depth 50)
run([sys.executable, "scripts/15_retrieval_recall.py", "--k", 1, 3, 5, 8,
     "--rerank", "--rerank-depth", 50])

In [ ]:
# Method 2: stronger multilingual embeddings.
# e5-base currently adds only ~2pts over BM25, which is why it is suspect.
run([sys.executable, "scripts/15_retrieval_recall.py", "--k", 1, 3, 5, 8,
     "--dense", "BAAI/bge-m3"])

In [ ]:
# Method 3: dense + rerank together (the full stack)
run([sys.executable, "scripts/15_retrieval_recall.py", "--k", 1, 3, 5, 8,
     "--dense", "BAAI/bge-m3", "--rerank", "--rerank-depth", 50])

## Verdict — judged on never-audited records only

In [ ]:
import glob, json, shutil, pathlib

rows = []
for path in sorted(glob.glob("reports/retrieval_recall*.json")):
    d = json.load(open(path))
    cell = d.get("recall", {}).get("k=8", {})
    rows.append((pathlib.Path(path).stem.replace("retrieval_recall", "") or "_bm25",
                 cell.get("full_hit"), cell.get("full_hit_never_audited")))

print(f"{'config':<22}{'all (tuned-on)':>16}{'NEVER AUDITED':>16}")
for name, all_hit, clean in rows:
    a = f"{all_hit:.1%}" if all_hit is not None else "-"
    c = f"{clean:.1%}" if clean is not None else "-"
    print(f"{name:<22}{a:>16}{c:>16}")

print("\nOnly the right-hand column is evidence. The synonym table scored")
print("+16pts on the left and +0.9 on the right — that is the failure mode")
print("this comparison exists to catch.")

out = pathlib.Path("/kaggle/working/nyaya-retrieval-results")
out.mkdir(exist_ok=True)
for path in glob.glob("reports/retrieval_recall*.json"):
    shutil.copy(path, out)
for path in glob.glob("data/generated/rerank_cache.json"):
    shutil.copy(path, out)
shutil.make_archive("/kaggle/working/nyaya-retrieval-results", "zip", out)
print("\nDownload nyaya-retrieval-results.zip from the Output tab.")